In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

C:\Users\SUYASH\AppData\Roaming\Python\Python311\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [13]:
X = np.array([
        [200,13],
        [200,15],
        [200,17],
        [250,13],
        [250,15],
        [250,17]
    ])
y = np.array([1,0,0,1,0,0])

In [25]:
print(X.shape)
print(y.shape)

(6, 2)
(6,)


Normalizing Data:
Fitting the weights to the data will proceed more quickly if the data is normalized. This is the same procedure, where features in the data is normalized to have a similar range. 

In [15]:
print(f"Temparature Max, Min pre normalization:{np.max(X[:,0]):0.2f},{np.min(X[:,0]):0.2f}")
print(f"Duration Max, Min pre normalization:{np.max(X[:,1]):0.2f},{np.min(X[:,1]):0.2f}")

norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X) #Learns mean, variance
Xn = norm_l(X)

print(f"Temparature Max, Min post normalization:{np.max(Xn[:,0]):0.2f},{np.min(Xn[:,0]):0.2f}")
print(f"Duration Max, Min post normalization:{np.max(Xn[:,1]):0.2f},{np.min(Xn[:,1]):0.2f}")


Temparature Max, Min pre normalization:250.00,200.00
Duration Max, Min pre normalization:17.00,13.00
Temparature Max, Min post normalization:1.00,-1.00
Duration Max, Min post normalization:1.22,-1.22


In [27]:
print(Xn.shape)
print(y.shape)

(6, 2)
(6,)


In [23]:
#Tile/Copy our data to increase the training set size and reduce the number of training epochs
Xt = np.tile(Xn, (1000,1))
Yt = np.tile(y,(1000,1))
print(Xt.shape,Yt.shape)

(6000, 2) (1000, 6)


In [17]:
tf.random.set_seed(1234) #applied to achieve consistent results
model = Sequential(
    [
        tf.keras.Input(shape=(2,)),
        Dense(3, activation='sigmoid', name='layer1'),
        Dense(1, activation='sigmoid', name='layer2')
    ]
)

Note 1: The tf.keras.Input(shape=(2,)), specifies the expected shape of the input. This allows Tensorflow to size the weights and bias parameters at this point. This is useful when exploring Tensorflow models. This statement can be omitted in practice and Tensorflow will size the network parameters when the input data is specified in the model.fit statement.

Note 2: Including the sigmoid activation in the final layer is not considered best practice. It would instead be accounted for in the loss which improves numerical stability. 

In [ ]:
# Define the activation function
# g = sigmoid

In [ ]:
# def my_dense(a_in, W, b):
#     """
#     Computes dense layer
#     Args:
#       a_in (ndarray (n, )) : Data, 1 example 
#       W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
#       b    (ndarray (j, )) : bias vector, j units  
#     Returns
#       a_out (ndarray (j,))  : j units|
#     """
#     units = W.shape[1]
#     a_out = np.zeros(units)
#     for j in range(units):               
#         w = W[:,j]                                    
#         z = np.dot(w, a_in) + b[j]         
#         a_out[j] = g(z)               
#     return(a_out)

In [ ]:
# def my_sequential(x, W1, b1, W2, b2):
#     a1 = my_dense(x,  W1, b1)
#     a2 = my_dense(a1, W2, b2)
#     return(a2)

In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Dense)                  │ (None, 3)              │             9 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Dense)                  │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13 (52.00 B)

 Trainable params: 13 (52.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
L1_num_params = 2 * 3 + 3 #W1 parameters + b1 parameters
L2_num_params = 3 * 1 + 1 #W2 parameters + b2 parameters
print("L1 params:",L1_num_params," L2 params:",L2_num_params)

L1 params: 9  L2 params: 4


The weights  𝑊 should be of size (number of features in input, number of units in the layer) 
while the bias  𝑏 size should match the number of units in the layer:

In the first layer with 3 units, we expect W to have a size of (2,3) and  𝑏
  should have 3 elements.
In the second layer with 1 unit, we expect W to have a size of (3,1) and  𝑏
  should have 1 element.

In [20]:
W1, b1 = model.get_layer('layer1').get_weights()
W2, b2 = model.get_layer('layer2').get_weights()
print(f"W1{W1.shape}:\n", W1, f"\nb1{b1.shape}:", b1)
print(f"W2{W2.shape}:\n", W2, f"\nb2{b2.shape}:", b2)

W1(2, 3):
 [[-0.06229639  0.06699336 -1.0308223 ]
 [-0.8688131   0.21406412  0.24677145]] 
b1(3,): [0. 0. 0.]
W2(3, 1):
 [[0.5547699]
 [0.3165164]
 [0.7383076]] 
b2(1,): [0.]


In [ ]:
# W1_tmp = np.array( [[-8.93,  0.29, 12.9 ], [-0.1,  -7.32, 10.81]] )
# b1_tmp = np.array( [-9.82, -9.28,  0.96] )
# W2_tmp = np.array( [[-31.18], [-27.59], [-32.56]] )
# b2_tmp = np.array( [15.41] )

In [29]:
model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
)

model.fit(
    Xt,Yt.flatten(),            
    epochs=10,
)

Epoch 1/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.5363
Epoch 2/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1514
Epoch 3/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0482
Epoch 4/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0237
Epoch 5/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0143
Epoch 6/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0097
Epoch 7/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0070
Epoch 8/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0053
Epoch 9/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0042
Epoch 10/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0034


In [ ]:
# def my_predict(X, W1, b1, W2, b2):
#     m = X.shape[0]
#     p = np.zeros((m,1))
#     for i in range(m):
#         p[i,0] = my_sequential(X[i], W1, b1, W2, b2)
#     return(p)

In [30]:
W1, b1 = model.get_layer('layer1').get_weights()
W2, b2 = model.get_layer('layer2').get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

W1:
 [[ 1.6507725e-03 -2.2031330e-03  2.9398522e-03]
 [-5.1806011e+00  4.8134909e+00 -3.9256773e+00]] 
b1: [-2.9701185  2.7962492 -2.1979692]
W2:
 [[ 4.807897 ]
 [-4.665266 ]
 [ 3.1037936]] 
b2: [-1.7327325]


In [ ]:
#Testing our model using sample test data of 2 features.
x_test = np.array([
    [200,13.9], #positive example
    [200,17] #negative example
]
)

x_testn = norm_l(x_test) #normalize test data

predictions = model.predict(x_testn) #prediction

print("Predictions:\n",predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Predictions:
 [[0.79372185]
 [0.0016688 ]]


In [33]:
#To convert the probabilities to a decision, we apply a threshold
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i]>=0.5:
        yhat[i] = 1
    else:
        yhat[i]=0
        
print(f"Decisions: \n {yhat}")           

Decisions: 
 [[1.]
 [0.]]
